# Notebook 3 — Metadata Analysis

**Project:** P63 – Multimodal Deep Learning for Autoimmune Disease Diagnosis  
**Phase:** Rheumatoid Arthritis (RA) only  
**Dataset:** RAM-H1200-v1  

---

## What this notebook does

This is the **third notebook** in the pipeline. It performs a thorough analysis of the `Metadata.xlsx` file — the clinical and acquisition metadata that accompanies the images.  

Questions answered:

1. What are all the available columns?
2. Which columns are numerical? Which are categorical?
3. Are there missing values? If so, where and how many?
4. Are there duplicate records?
5. What do the distributions of each variable look like?
6. How do clinical variables (Age, Sex) relate to the RA label?
7. What are the inconsistencies in columns like `PixelSpacing` and `ImageSize`?
8. What features are safe to use for ML? Which ones risk target leakage?
9. How should missing values be handled?

**Prerequisite:** Run `01_dataset_overview.ipynb` first.  

**Outputs saved:**
- `outputs/reports/03_metadata_missing.csv` — missing value report
- `outputs/reports/03_metadata_summary.csv` — per-column summary
- Several plots in `outputs/plots/`

---
## 0. Imports and configuration

In [ ]:
import os
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
sns.set_theme(style='whitegrid', palette='muted')

print('All imports successful.')

In [ ]:
PROJECT_ROOT  = Path(os.getcwd()).parent
METADATA_FILE = PROJECT_ROOT / 'Dataset' / 'ra' / 'Metadata.xlsx'
MANIFEST_CSV  = PROJECT_ROOT / 'outputs' / 'reports' / 'image_manifest.csv'
OUTPUTS_DIR   = PROJECT_ROOT / 'outputs' / 'reports'
PLOTS_DIR     = PROJECT_ROOT / 'outputs' / 'plots'

assert METADATA_FILE.exists(), f'Metadata file not found: {METADATA_FILE}'
assert MANIFEST_CSV.exists(),  f'image_manifest.csv not found. Run Notebook 01 first.'

print('Paths OK.')

---
## 1. Load metadata and manifest

In [ ]:
# ── Raw metadata (one row per exam/case) ──────────────────────────────────────
meta = pd.read_excel(METADATA_FILE)
print(f'Raw metadata shape: {meta.shape}')
print(f'Columns: {list(meta.columns)}')
print()
meta.head(5)

In [ ]:
# ── Manifest (per-image, from Notebook 01) ────────────────────────────────────
manifest = pd.read_csv(MANIFEST_CSV)
print(f'Manifest shape: {manifest.shape}')

# Restrict metadata to rows that actually have images on disk
stems_on_disk = set(manifest['base_stem'].unique())
meta_active = meta[meta['Mapped Image Stem'].isin(stems_on_disk)].copy()
print(f'Metadata rows with images on disk: {len(meta_active)}')
print(f'Metadata rows without images     : {len(meta) - len(meta_active)}')
print()
print('We will analyse BOTH the full metadata AND the active (imaged) subset.')

---
## 2. Column inventory — numerical vs categorical

In [ ]:
# ── Classify each column ──────────────────────────────────────────────────────
# We inspect dtypes and unique counts to decide the type of each column.

col_info = []
for col in meta.columns:
    dtype    = str(meta[col].dtype)
    n_unique = meta[col].nunique()
    n_null   = meta[col].isnull().sum()
    sample   = str(meta[col].dropna().iloc[0]) if n_null < len(meta) else 'ALL NULL'

    # Determine our semantic type
    if col in ['isRA']:
        sem_type = 'TARGET (binary)'
    elif col in ['Normalized PatientID', 'StudyID']:
        sem_type = 'IDENTIFIER (do not use as feature)'
    elif col in ['Mapped Image Stem']:
        sem_type = 'KEY (filename stem)'
    elif dtype in ['float64', 'int64'] and n_unique > 10:
        sem_type = 'Numerical (continuous)'
    elif dtype in ['int64'] and n_unique <= 10:
        sem_type = 'Numerical (discrete)'
    else:
        sem_type = 'Categorical'

    col_info.append({
        'column'          : col,
        'dtype'           : dtype,
        'n_unique'        : n_unique,
        'n_missing'       : n_null,
        'missing_pct'     : round(n_null / len(meta) * 100, 2),
        'sample_value'    : sample,
        'semantic_type'   : sem_type,
    })

col_df = pd.DataFrame(col_info)
print('Column inventory:')
print(col_df.to_string(index=False))

---
## 3. Missing values — full report

In [ ]:
# ── Missing value count and percentage ────────────────────────────────────────
missing = meta.isnull().sum().reset_index()
missing.columns = ['column', 'n_missing']
missing['missing_pct'] = (missing['n_missing'] / len(meta) * 100).round(2)
missing = missing.sort_values('n_missing', ascending=False)

print('Missing values per column:')
print(missing.to_string(index=False))

missing.to_csv(OUTPUTS_DIR / '03_metadata_missing.csv', index=False)
print(f'\nMissing value report saved → {OUTPUTS_DIR / "03_metadata_missing.csv"}')

In [ ]:
# ── Duplicate rows check ──────────────────────────────────────────────────────
n_dup_full = meta.duplicated().sum()
n_dup_stem = meta.duplicated(subset='Mapped Image Stem').sum()

print(f'Fully duplicate rows               : {n_dup_full}')
print(f'Rows with duplicate Mapped Image Stem: {n_dup_stem}')

if n_dup_stem > 0:
    print()
    dups = meta[meta.duplicated(subset='Mapped Image Stem', keep=False)]
    print(dups[['Mapped Image Stem', 'isRA', 'Age', 'Sex']].to_string(index=False))

---
## 4. Target variable analysis — `isRA`

In [ ]:
label_map = {0: 'Non-RA', 1: 'RA'}

# ── Full metadata (all 836 rows) ──────────────────────────────────────────────
print('isRA distribution — full metadata (836 rows, incl. rows without images):')
vc_full = meta['isRA'].map(label_map).value_counts()
print(vc_full)
print()

# ── Active metadata (790 rows with images on disk) ────────────────────────────
print('isRA distribution — active metadata (rows with images on disk):')
vc_active = meta_active['isRA'].map(label_map).value_counts()
print(vc_active)
print()

# Imbalance ratio
ratio = vc_active.get('RA', 0) / max(vc_active.get('Non-RA', 1), 1)
print(f'Imbalance ratio (RA : Non-RA) ≈ {ratio:.1f} : 1')
print()
print('IMPORTANT: This is a severe class imbalance.')
print('During model training, use class-weighted loss or oversampling for the minority class.')

---
## 5. Numerical feature analysis — `Age`

In [ ]:
# ── Age statistics ─────────────────────────────────────────────────────────────
print('Age — overall statistics:')
print(meta['Age'].describe().round(2))
print()
print('Age — by isRA class:')
print(meta.groupby('isRA')['Age'].describe().round(2))

In [ ]:
# ── Age outlier check ─────────────────────────────────────────────────────────
# IQR method: anything below Q1 - 1.5*IQR or above Q3 + 1.5*IQR is an outlier
Q1  = meta['Age'].quantile(0.25)
Q3  = meta['Age'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

age_outliers = meta[(meta['Age'] < lower_bound) | (meta['Age'] > upper_bound)]
print(f'IQR bounds for Age: [{lower_bound:.1f}, {upper_bound:.1f}]')
print(f'Age outliers (outside IQR bounds): {len(age_outliers)}')
if not age_outliers.empty:
    print(age_outliers[['Mapped Image Stem', 'Age', 'isRA']].to_string(index=False))

---
## 6. Categorical feature analysis

In [ ]:
# ── Sex ───────────────────────────────────────────────────────────────────────
print('Sex distribution (full metadata):')
print(meta['Sex'].value_counts())
print()

# Cross-tab: Sex × isRA
print('Sex × isRA cross-table:')
ct_sex = pd.crosstab(meta['Sex'], meta['isRA'].map(label_map))
ct_sex['Total'] = ct_sex.sum(axis=1)
ct_sex['RA_pct'] = (ct_sex.get('RA', 0) / ct_sex['Total'] * 100).round(1)
print(ct_sex)
print()
print('Note: "O" in the Sex column likely represents "Other" or an anonymisation category.')

In [ ]:
# ── Center ─────────────────────────────────────────────────────────────────────
print('Center distribution:')
print(meta['Center'].value_counts())
print()

# Cross-tab: Center × isRA
print('Center × isRA cross-table:')
ct_center = pd.crosstab(meta['Center'], meta['isRA'].map(label_map))
ct_center['Total']  = ct_center.sum(axis=1)
ct_center['RA_pct'] = (ct_center.get('RA', 0) / ct_center['Total'] * 100).round(1)
print(ct_center)

In [ ]:
# ── LR column analysis ────────────────────────────────────────────────────────
# This column encodes laterality in the metadata.
# The image filename also encodes it (suffix _L or _R).
print('LR column unique values and counts:')
print(meta['LR'].value_counts())
print()
print('Interpretation:')
print('  LR  = both left and right hand captured in one exam entry')
print('  L   = only left hand')
print('  R   = only right hand')
print()
print('The LR column from metadata and the laterality in the filename')
print('both carry laterality information; we use the filename suffix as the')
print('per-image ground truth (more granular).')

In [ ]:
# ── PixelSpacing — raw value inspection ───────────────────────────────────────
# This column is stored as a string like '[0.15, 0.15]' rather than a number.
# We will parse it in Notebook 04 (preprocessing).

print('PixelSpacing unique values:')
ps_counts = meta['PixelSpacing'].value_counts()
print(ps_counts)
print()
print('PixelSpacing × isRA:')
ct_ps = pd.crosstab(meta['PixelSpacing'], meta['isRA'].map(label_map))
ct_ps['Total'] = ct_ps.sum(axis=1)
print(ct_ps)
print()
print('ISSUE: PixelSpacing is stored as a string, not a float.')
print('This will be parsed into a numeric value in the preprocessing notebook.')

In [ ]:
# ── ImageSize — raw value inspection ──────────────────────────────────────────
# Also stored inconsistently: '1670x2010', '[1670, 2010]', etc.

print('ImageSize unique values:')
print(meta['ImageSize'].value_counts())
print()
print('ISSUE: ImageSize is stored in multiple string formats.')
print('This will be standardised in the preprocessing notebook.')

In [ ]:
# ── StudyID ───────────────────────────────────────────────────────────────────
print('StudyID unique values:')
print(sorted(meta['StudyID'].unique()))
print(f'Count: {meta["StudyID"].nunique()}')
print()
print('StudyID appears to be a per-exam sequential ID.')
print('It is NOT a patient-level identifier and should NOT be used as a feature')
print('because it could encode information about the split assignment.')

---
## 7. Feature usability assessment

Not all columns are safe or useful as input features for a machine learning model.  
We classify each column here.

In [ ]:
feature_assessment = [
    {
        'column'       : 'Mapped Image Stem',
        'use_as_feature': 'NO',
        'reason'       : 'Primary key / filename — purely an identifier, no predictive value.'
    },
    {
        'column'       : 'StudyID',
        'use_as_feature': 'NO',
        'reason'       : 'Sequential exam ID — could leak split order, no clinical meaning.'
    },
    {
        'column'       : 'Normalized PatientID',
        'use_as_feature': 'NO — use for stratified splitting only',
        'reason'       : 'Patient ID — must be used to prevent data leakage across splits. NOT a feature.'
    },
    {
        'column'       : 'isRA',
        'use_as_feature': 'TARGET LABEL',
        'reason'       : 'Binary label: 0=Non-RA, 1=RA. This is what we predict.'
    },
    {
        'column'       : 'Sex',
        'use_as_feature': 'YES (with encoding)',
        'reason'       : 'Clinical variable. Encode as numeric (F=0, M=1, O=2 or one-hot).'
    },
    {
        'column'       : 'Age',
        'use_as_feature': 'YES (after normalisation)',
        'reason'       : 'Clinical variable. Continuous float. Normalise with training-set stats.'
    },
    {
        'column'       : 'Center',
        'use_as_feature': 'CAUTION — site effect',
        'reason'       : 'Imaging centre. Could introduce site-bias. Use with caution or exclude.'
    },
    {
        'column'       : 'PixelSpacing',
        'use_as_feature': 'YES (after parsing)',
        'reason'       : 'Acquisition parameter. Parse string to float. Useful for scale normalisation.'
    },
    {
        'column'       : 'ImageSize',
        'use_as_feature': 'DERIVED — see notes',
        'reason'       : 'Original image dimensions. Useful to compute physical size but redundant after resize.'
    },
    {
        'column'       : 'LR',
        'use_as_feature': 'YES (with encoding)',
        'reason'       : 'Laterality from metadata. Corroborated by filename suffix (_L/_R). Encode as numeric.'
    },
]

fa_df = pd.DataFrame(feature_assessment)
print('Feature usability assessment:')
print(fa_df.to_string(index=False))

---
## 8. Visualisations

In [ ]:
# ── Figure 1: key categorical distributions ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Metadata — Categorical Feature Distributions', fontsize=13, fontweight='bold')

palette = {'Non-RA': '#DD8452', 'RA': '#4C72B0'}

# Plot 1: Sex × isRA
sex_plot = meta.groupby(['Sex', 'isRA']).size().reset_index(name='count')
sex_plot['label'] = sex_plot['isRA'].map(label_map)
sex_pivot = sex_plot.pivot(index='Sex', columns='label', values='count').fillna(0)
sex_pivot.plot(kind='bar', ax=axes[0], color=[palette.get(c, 'grey') for c in sex_pivot.columns],
               edgecolor='white', rot=0)
axes[0].set_title('Sex × Class')
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Sex')

# Plot 2: Center × isRA
ctr_plot = meta.groupby(['Center', 'isRA']).size().reset_index(name='count')
ctr_plot['label'] = ctr_plot['isRA'].map(label_map)
ctr_pivot = ctr_plot.pivot(index='Center', columns='label', values='count').fillna(0)
ctr_pivot.plot(kind='bar', ax=axes[1], color=[palette.get(c, 'grey') for c in ctr_pivot.columns],
               edgecolor='white', rot=30)
axes[1].set_title('Imaging Centre × Class')
axes[1].set_ylabel('Count')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', labelsize=9)

# Plot 3: LR × isRA
lr_plot = meta.groupby(['LR', 'isRA']).size().reset_index(name='count')
lr_plot['label'] = lr_plot['isRA'].map(label_map)
lr_pivot = lr_plot.pivot(index='LR', columns='label', values='count').fillna(0)
lr_pivot.plot(kind='bar', ax=axes[2], color=[palette.get(c, 'grey') for c in lr_pivot.columns],
              edgecolor='white', rot=0)
axes[2].set_title('Laterality (LR) × Class')
axes[2].set_ylabel('Count')
axes[2].set_xlabel('LR value')

plt.tight_layout()
save_path = PLOTS_DIR / '03_categorical_distributions.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved → {save_path}')

In [ ]:
# ── Figure 2: Age distribution by class ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Age Distribution by RA Class', fontsize=13, fontweight='bold')

# Histogram
meta['label'] = meta['isRA'].map(label_map)
for label, grp in meta.groupby('label'):
    axes[0].hist(grp['Age'].dropna(), bins=25, alpha=0.6,
                 label=label, color=palette[label], density=True)
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Density')
axes[0].set_title('Age histogram')
axes[0].legend()

# Boxplot
groups_age  = [meta[meta['label'] == lbl]['Age'].dropna().values for lbl in ['Non-RA', 'RA']]
bp = axes[1].boxplot(groups_age, labels=['Non-RA', 'RA'], patch_artist=True,
                     medianprops=dict(color='black', linewidth=2))
bp['boxes'][0].set_facecolor('#DD8452')
bp['boxes'][1].set_facecolor('#4C72B0')
axes[1].set_ylabel('Age (years)')
axes[1].set_title('Age box plot')

plt.tight_layout()
save_path = PLOTS_DIR / '03_age_distribution.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved → {save_path}')

In [ ]:
# ── Figure 3: PixelSpacing distribution by class ──────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

ps_class = meta.groupby(['PixelSpacing', 'isRA']).size().reset_index(name='count')
ps_class['label'] = ps_class['isRA'].map(label_map)
ps_pivot = ps_class.pivot(index='PixelSpacing', columns='label', values='count').fillna(0)
ps_pivot.plot(kind='bar', ax=ax,
              color=[palette.get(c, 'grey') for c in ps_pivot.columns],
              edgecolor='white', rot=0)
ax.set_title('PixelSpacing × Class')
ax.set_ylabel('Count')
ax.set_xlabel('PixelSpacing value (mm/pixel)')

plt.tight_layout()
save_path = PLOTS_DIR / '03_pixelspacing_distribution.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved → {save_path}')

In [ ]:
# ── Figure 4: Patient image count distribution ────────────────────────────────
pt_img_counts = manifest.groupby('Normalized PatientID')['filename'].count().reset_index()
pt_img_counts.columns = ['Normalized PatientID', 'n_images']

fig, ax = plt.subplots(figsize=(9, 4))
pt_img_counts['n_images'].value_counts().sort_index().plot(kind='bar', ax=ax,
    color='#8172B2', edgecolor='white')
ax.set_title('Distribution of Images per Patient')
ax.set_xlabel('Number of images per patient')
ax.set_ylabel('Number of patients')
ax.tick_params(axis='x', rotation=0)
for p in ax.patches:
    ax.annotate(str(int(p.get_height())),
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
save_path = PLOTS_DIR / '03_images_per_patient.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved → {save_path}')

---
## 9. Save the metadata summary

In [ ]:
# Build a concise per-column summary for the report
summary_rows = []

for col in meta.columns:
    if col == 'label':
        continue
    dtype  = str(meta[col].dtype)
    n_miss = int(meta[col].isnull().sum())
    n_uniq = int(meta[col].nunique())

    if dtype in ['float64', 'int64']:
        stats = meta[col].describe()
        details = (f'min={stats["min"]:.2f}, mean={stats["mean"]:.2f}, '
                   f'max={stats["max"]:.2f}, std={stats["std"]:.2f}')
    else:
        top_vals = meta[col].value_counts().head(3)
        details  = ', '.join([f'{v}:{c}' for v, c in top_vals.items()])

    summary_rows.append({
        'column'        : col,
        'dtype'         : dtype,
        'n_missing'     : n_miss,
        'missing_pct'   : round(n_miss / len(meta) * 100, 2),
        'n_unique'      : n_uniq,
        'summary_stats' : details,
    })

meta_summary_df = pd.DataFrame(summary_rows)
print('Metadata column summary:')
print(meta_summary_df.to_string(index=False))

meta_summary_df.to_csv(OUTPUTS_DIR / '03_metadata_summary.csv', index=False)
print(f'\nMetadata summary saved → {OUTPUTS_DIR / "03_metadata_summary.csv"}')

---
## Summary of findings

| Column | Type | Notes |
|---|---|---|
| `Mapped Image Stem` | Key | Links metadata row to image filename |
| `StudyID` | Identifier | Sequential — do NOT use as feature |
| `Normalized PatientID` | Identifier | Use for patient-stratified splitting only |
| `isRA` | **Target label** | Binary: 0=Non-RA, 1=RA |
| `Sex` | Categorical | F / M / O. Encode numerically |
| `Age` | Numerical (continuous) | Float. Normalise with training-set mean & std |
| `Center` | Categorical | Imaging site — potential confound, use with caution |
| `PixelSpacing` | Categorical → parse to float | Stored as string `[x, x]`. Parse → `x` |
| `ImageSize` | Mixed string format | Multiple formats. Parse or drop |
| `LR` | Categorical | Laterality. Encode numerically |

**Missing values:** None in the full metadata file (every column is complete).  
**Duplicates:** Inspect output above.  
**Class imbalance:** ~14:1 (RA : Non-RA) — needs addressing during training.  
**Key risk:** 46 metadata rows have no image on disk (these rows should be excluded from preprocessing).  

Proceed to **Notebook 4** for the full preprocessing and dataset preparation pipeline.